# What this file does
- Influencial complaint analysis based on manual filtering

# Dependencies
#### Run the following file(s) before running this code.
- 03_baseline_similarity_graph.ipynb (or 03b (but not 03c))
- 07_gds_Louvain_Summary.ipynb
- 21_gds_Centrality_on-graph.ipynb
- 23_gds_Company.ipynb

##### To add some nodes, this code is updating the data in Mongo, and then updating neo4j accordingly.

##### Note:
URL of the Neo4j browser:
- https://[IP address]:7473/browser/

ID & Pass: 
- Use the one in .env


In [1]:
# Config
SAMPLING:bool       = True
NUM_SAMPLE:int      = 250   # Number of sample data to be ingested to the graph database
SUMMARY_SAMPLE:int  = 20    # Number of samples as inputs of summarizing
RAND_SEED:int       = 77    # Seed for sampling
NUM_SIM:int         = 3     # Number of results from KNN search (does not include the own node)
EMBEDDING_MODEL:str = "text-embedding-3-small"
MAX_TOKENS:int      = 7800  # Max 8192 - some safety buffer about 5%
INDEX_NAME:str      = "idx:complaints_vss"
FILE_PATH:str       = "../data/original/complaints-2025-11-02_04_18.csv"

In [2]:
import time
from datetime import datetime, timedelta

In [3]:
import os
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display
import tiktoken
import textwrap
import logging
import seaborn as sns

logger = logging.getLogger("neo4j")
logger.setLevel(logging.CRITICAL)

In [4]:
from dotenv import load_dotenv  
load_dotenv()

True

In [5]:
import neo4j

In [6]:
# Ignore unclosed SSL socket warnings - optional in case you get these errors
import warnings

In [7]:
warnings.filterwarnings(action="ignore", message="unclosed", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning) 

In [8]:
# Show all columns
pd.set_option('display.max_columns', None)

# Show all rows
pd.set_option('display.max_rows', None)

In [9]:
# Timestamp (Start)
current_datetime = datetime.now()
formatted_time = current_datetime.strftime("%Y-%m-%d_%H:%M:%S")
print(formatted_time)

start_time = time.time()

2026-01-08_22:09:03


### Neo4j

In [10]:
driver = neo4j.GraphDatabase.driver(
    uri=os.environ.get("NEO4J_URI"), 
    auth=(os.environ.get("NEO4J_USERNAME"), 
          os.environ.get("NEO4J_PASSWORD"))
)

In [11]:
session = driver.session(database="neo4j")

In [12]:
def my_neo4j_run_query_pandas(query, **kwargs):
    "run a query and return the results in a pandas dataframe"
    
    result = session.run(query, **kwargs)
    
    df = pd.DataFrame([r.values() for r in result], columns=result.keys())
    
    return df

# Market Analysis
In our next company-based complaint analysis, we would like to conduct a market assessment by investigating which companies share similar or identical complaint groups. With this analysis, we aim to answer the question: 'Which companies are structurally similar based on the problems they face?'

Please note that we may observe a higher density of complaint categories among the companies receiving the most complaints (TransUnion Intermediate Holdings, Inc., Equifax, Inc., and Experian Information Solutions Inc.). This means the number of complaints is not uniformly distributed among the companies included in our graph.

In [13]:
query = """
    match (c1:Company)<-[:COMPLAIN_TO]-(comp1:Complaint)-[s:SIMILAR]-(comp2:Complaint)-[:COMPLAIN_TO]->(c2:Company)
    
    // preventing duplicates
    where elementId(c1) < elementId(c2)
    
    // checking for score higher than 0.6
    and s.similarity_score > 0.6
    
    // counting the matches
    with c1, c2, count(s) as shared_vol, avg(s.similarity_score) as avg_score
    
    // ignoring the ones that only happened once
    where shared_vol > 1 

    return 
        c1.name as Company_1, 
        c2.name as Company_2, 
        shared_vol as Shared_Complaint_Volume,
        round(avg_score, 3) as Similarity_Score
    order by shared_vol desc
    limit 15
"""

#opening the connection and running the query directly
with driver.session() as session:
    result = session.run(query)
    
    #turning the result into a list and printing in a df from
    data = [record.data() for record in result]
    df_similarity = pd.DataFrame(data)
    print(df_similarity)

                                Company_1  \
0  TRANSUNION INTERMEDIATE HOLDINGS, INC.   
1     Experian Information Solutions Inc.   
2  TRANSUNION INTERMEDIATE HOLDINGS, INC.   
3                           EQUIFAX, INC.   
4     Experian Information Solutions Inc.   
5                           EQUIFAX, INC.   

                             Company_2  Shared_Complaint_Volume  \
0  Experian Information Solutions Inc.                      115   
1                        EQUIFAX, INC.                       96   
2                        EQUIFAX, INC.                       92   
3            ENCORE CAPITAL GROUP INC.                        5   
4            ENCORE CAPITAL GROUP INC.                        3   
5                      CL Holdings LLC                        2   

   Similarity_Score  
0             0.895  
1             0.874  
2             0.890  
3             0.839  
4             0.823  
5             0.911  


From the complaint similarity score results above, we can see that companies like TRANSUNION INTERMEDIATE HOLDINGS, INC. and Experian Information Solutions Inc. show up often. This is likely because these companies provide services to many other organizations; consequently, they receive more complaints and appear frequently across the dataset.

In [14]:
def analyze_specific_company(tx, target_name):
    query = """
    // finding the company that has been used as input
    match (target:Company)<-[:COMPLAIN_TO]-(my_complaint:Complaint)
    
    // looking for other complaints that are similar to ours and seeing which company they belong to
    match (my_complaint)-[s:SIMILAR]-(their_complaint:Complaint)-[:COMPLAIN_TO]->(other_company:Company)
    
    // making sure the name matches even if the user typed uppercase or lowercase
    where toLower(target.name) contains toLower($input_name)
    
    // preventing it from matching itself
    and elementId(target) <> elementId(other_company) 
    
    // checking the similarity score above 0.5
    and s.similarity_score > 0.5
    
    // removing the debt collector companies because they show up everywhere
    // since they share similar complaints to those they provide service
    and not toLower(other_company.name) contains 'portfolio recovery'
    and not toLower(other_company.name) contains 'transworld'
    and not toLower(other_company.name) contains 'resurgent'

    // counting how many times they match and getting the average score
    with other_company, count(s) as shared_vol, avg(s.similarity_score) as avg_score
    
    // ignoring it if they only matched once because that could be random
    where shared_vol > 1 

    // returning the list of companies and the scores
    return 
        other_company.name as Similar_Company,
        shared_vol as Shared_Complaint_Volume,
        round(avg_score, 3) as Avg_Similarity
    order by shared_vol desc
    limit 20
    """
    
    result = tx.run(query, input_name=target_name)
    return [record.data() for record in result]

#Input Area
target_input = "CITIBANK, N.A."  # a company name here from the company column in the df

print(f"Analyzing companies similar to: '{target_input}'...\n")

with driver.session() as session:
    # running the function we wrote above
    data = session.execute_read(analyze_specific_company, target_input)
    df_tool = pd.DataFrame(data)
    
    if df_tool.empty:
        print("No similar companies found. (Check spelling or lower the score threshold)")
    else:
        pd.set_option('display.max_columns', None)
        pd.set_option('display.width', 1000)
        print(df_tool)

Analyzing companies similar to: 'CITIBANK, N.A.'...

No similar companies found. (Check spelling or lower the score threshold)


From this brief analysis, we see that our dataset spans five main industry groups. The first is credit reporting, which includes TransUnion, Equifax, and Experian which are identified as outliers due to the number of complaint they are paired with in our data set. The second group is debt collection, represented by companies such as Transworld, Resurgent, Portfolio Recovery, Encore, and IC System. We also observe a group of traditional banking institutions, including Wells Fargo, Bank of America, Citibank, US Bancorp, and Webster Bank. In addition, there is a noticeable presence of crypto and fintech companies, such as Chime, Coinbase, Robinhood, PayPal, and Fig Tech. Finally, the dataset also reflects companies in the auto loan sector, including DriveTime, Hyundai Capital, and Southern Auto Finance.


In [15]:
def analyze_specific_company(tx, target_name):
    query = """
    // finding the company that has been used as input
    match (target:Company)<-[:COMPLAIN_TO]-(my_complaint:Complaint)
    
    // looking for other complaints that are similar to ours and seeing which company they belong to
    match (my_complaint)-[s:SIMILAR]-(their_complaint:Complaint)-[:COMPLAIN_TO]->(other_company:Company)
    
    // making sure the name matches even if the user typed uppercase or lowercase
    where toLower(target.name) contains toLower($input_name)
    
    // preventing it from matching itself
    and elementId(target) <> elementId(other_company) 
    
    // checking the similarity score above 0.5
    and s.similarity_score > 0.5
    
    // removing the debt collector companies because they show up everywhere
    // since they share similar complaints to those they provide service
    and not toLower(other_company.name) contains 'portfolio recovery'
    and not toLower(other_company.name) contains 'transworld'
    and not toLower(other_company.name) contains 'resurgent'

    // counting how many times they match and getting the average score
    with other_company, count(s) as shared_vol, avg(s.similarity_score) as avg_score
    
    // ignoring it if they only matched once because that could be random
    where shared_vol > 1 

    // returning the list of companies and the scores
    return 
        other_company.name as Similar_Company,
        shared_vol as Shared_Complaint_Volume,
        round(avg_score, 3) as Avg_Similarity
    order by shared_vol desc
    limit 20
    """
    
    result = tx.run(query, input_name=target_name)
    return [record.data() for record in result]

#Input Area
target_input = "CAPITAL ONE FINANCIAL CORPORATION"  # a company name here from the company column in the df

print(f"Analyzing companies similar to: '{target_input}'...\n")

with driver.session() as session:
    #running the function we wrote above
    data = session.execute_read(analyze_specific_company, target_input)
    df_tool = pd.DataFrame(data)
    
    if df_tool.empty:
        print("No similar companies found. (Check spelling or lower the score threshold)")
    else:
        pd.set_option('display.max_columns', None)
        pd.set_option('display.width', 1000)
        print(df_tool)

Analyzing companies similar to: 'CAPITAL ONE FINANCIAL CORPORATION'...

No similar companies found. (Check spelling or lower the score threshold)


Due to the significant imbalance in complaint volume across companies, we are focusing our analysis on the most influential complaint nodes within each of our five distinct market domains.

## The most influential complaint among the credit reporting companies:

In [16]:
# def get_the_main_complaint(tx):
#     #ensure we don't overwrite any graph
#     tx.run("call gds.graph.drop('credit_graph', false)")

    
#     #loading the graph for credit reporting companies
#     tx.run("""
#     call gds.graph.project.cypher(
#       'credit_graph',
#       'match (c:Complaint)-[:COMPLAIN_TO]->(co:Company)
#        where toLower(co.name) contains "equifax"
#           or toLower(co.name) contains "transunion"
#           or toLower(co.name) contains "experian"
#        return id(c) as id',
#       'match (c1)-[s:SIMILAR]-(c2)
#        where s.similarity_score > 0.6
#        return id(c1) as source, id(c2) as target',
#       { validateRelationships: false }
#     )
#     """)
    
#     #running pagerank
#     result = tx.run("""
#     call gds.pageRank.stream('credit_graph')
#     yield nodeId, score
    
#     with gds.util.asNode(nodeId) as c, score
#     match (c)-[:COMPLAIN_TO]->(co:Company)
    
#     return 
#         co.name as Company,
        
#         coalesce(c.consumer_complaint_narrative, "No Complaint Found") as Complaint_Narrative,
        
#         round(score, 2) as Score
#     order by score desc
#     limit 1
#     """)
    
#     data = [record.data() for record in result]
    
#     #cleanup
#     tx.run("call gds.graph.drop('credit_graph', false)")
    
#     return data


# with driver.session() as session:
#     data = session.execute_read(get_the_main_complaint)
    
#     if not data:
#         print("didn't find anything.")
#     else:
#         row = data[0]
        
#         print("\n" + "+"*80)
#         print(f" COMPANY: {row['Company']}")
#         print("+" * 80)
#         print(" THE COMPLAINT NARRATIVE:")
#         print("+" * 80)
        
#         #textwrap makes long text readable in the terminal
#         wrapper = textwrap.TextWrapper(width=80)
#         print(wrapper.fill(text=row['Complaint_Narrative']))
        
#         print("+"*80)

### The most influential complaint among the debt collector companies:

In [17]:
def get_the_main_complaint(tx):
    #ensure we don't overwrite any graph
    tx.run("call gds.graph.drop('debt_graph', false)")

    
    #loading the graph for debt collector companies
    tx.run("""
    call gds.graph.project.cypher(
      'debt_graph',
      'match (c:Complaint)-[:COMPLAIN_TO]->(co:Company)
       where toLower(co.name) contains "transworld"
          or toLower(co.name) contains "resurgent"
          or toLower(co.name) contains "portfolio recovery"
          or toLower(co.name) contains "encore"
          or toLower(co.name) contains "ic system"
       return id(c) as id',
      'match (c1)-[s:SIMILAR]-(c2)
       where s.similarity_score > 0.6
       return id(c1) as source, id(c2) as target',
      { validateRelationships: false }
    )
    """)
    
    #running pagerank
    result = tx.run("""
    call gds.pageRank.stream('debt_graph')
    yield nodeId, score
    
    with gds.util.asNode(nodeId) as c, score
    match (c)-[:COMPLAIN_TO]->(co:Company)
    
    return 
        co.name as Company,
        
        coalesce(c.consumer_complaint_narrative, "No Complaint Found") as Complaint_Narrative,
        
        round(score, 2) as Score
    order by score desc
    limit 1
    """)
    
    data = [record.data() for record in result]
    
    #cleanup
    tx.run("call gds.graph.drop('debt_graph', false)")
    
    return data


with driver.session() as session:
    data = session.execute_read(get_the_main_complaint)
    
    if not data:
        print("didn't find anything.")
    else:
        row = data[0]
        
        print("\n" + "+"*80)
        print(f" COMPANY: {row['Company']}")
        print("+" * 80)
        print(" THE COMPLAINT NARRATIVE:")
        print("+" * 80)
        
        #textwrap makes long text readable in the terminal
        wrapper = textwrap.TextWrapper(width=80)
        print(wrapper.fill(text=row['Complaint_Narrative']))
        
        print("+"*80)


++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
 COMPANY: Portfolio Recovery Associates, LLC
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
 THE COMPLAINT NARRATIVE:
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
ACCOUNT : I am filing this formal complaint against a company that is currently
furnishing information to the credit reporting agencies regarding an alleged
debt associated with my name and credit profile. I previously submitted a
written request to this company demanding validation of the debt, including : A
copy of the original contract bearing my signature Documentation establishing
the chain of custody or assignment Any purchase agreement related to the alleged
debt As of todays date, the company has failed to respond to my request. Under
the Fair Debt Collection Practices Act ( FDCPA ), specifically 15 U.S. Code
1692g ( b ), I have the right to dispute the validity of a d

## The most influential complaint among the banks:

In [18]:
def get_the_main_complaint(tx):
    #ensure we don't overwrite any graph
    tx.run("call gds.graph.drop('bank_graph', false)")

    
    #loading the graph for banks
    tx.run("""
    call gds.graph.project.cypher(
      'bank_graph',
      'match (c:Complaint)-[:COMPLAIN_TO]->(co:Company)
       where toLower(co.name) contains "wells fargo"
          or toLower(co.name) contains "bank of america"
          or toLower(co.name) contains "citibank"
          or toLower(co.name) contains "us bancorp"
          or toLower(co.name) contains "webster bank"
       return id(c) as id',
      'match (c1)-[s:SIMILAR]-(c2)
       where s.similarity_score > 0.6
       return id(c1) as source, id(c2) as target',
      { validateRelationships: false }
    )
    """)
    
    #running pagerank
    result = tx.run("""
    call gds.pageRank.stream('bank_graph')
    yield nodeId, score
    
    with gds.util.asNode(nodeId) as c, score
    match (c)-[:COMPLAIN_TO]->(co:Company)
    
    return 
        co.name as Company,
        
        coalesce(c.consumer_complaint_narrative, "No Complaint Found") as Complaint_Narrative,
        
        round(score, 2) as Score
    order by score desc
    limit 1
    """)
    
    data = [record.data() for record in result]
    
    #cleanup
    tx.run("call gds.graph.drop('bank_graph', false)")
    
    return data


with driver.session() as session:
    data = session.execute_read(get_the_main_complaint)
    
    if not data:
        print("didn't find anything.")
    else:
        row = data[0]
        
        print("\n" + "+"*80)
        print(f" COMPANY: {row['Company']}")
        print("+" * 80)
        print(" THE COMPLAINT NARRATIVE:")
        print("+" * 80)
        
        #textwrap makes long text readable in the terminal
        wrapper = textwrap.TextWrapper(width=80)
        print(wrapper.fill(text=row['Complaint_Narrative']))
        
        print("+"*80)


++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
 COMPANY: CITIBANK, N.A.
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
 THE COMPLAINT NARRATIVE:
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
On //year>, Citibank sent me a notice that they will terminate my Citibank
account ( both my Checking AND Savings account ) due to violating the terms of
account per manual. When I reached out regarding what violation, they did not
give me a response nor any solutions despite multiple attempts I have been
waiting for my cheque to be mailed but have yet to receive it.
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++


## The most influential complaint among the crypto and fintech companies:

In [19]:
def get_the_main_complaint(tx):
    #ensure we don't overwrite any graph
    tx.run("call gds.graph.drop('crypto_graph', false)")

    
    #loading the graph for crypto and fintech companies
    tx.run("""
    call gds.graph.project.cypher(
      'crypto_graph',
      'match (c:Complaint)-[:COMPLAIN_TO]->(co:Company)
       where toLower(co.name) contains "chime"
          or toLower(co.name) contains "coinbase"
          or toLower(co.name) contains "robinhood"
          or toLower(co.name) contains "paypal"
          or toLower(co.name) contains "fig tech"
       return id(c) as id',
      'match (c1)-[s:SIMILAR]-(c2)
       where s.similarity_score > 0.6
       return id(c1) as source, id(c2) as target',
      { validateRelationships: false }
    )
    """)
    
    #running pagerank
    result = tx.run("""
    call gds.pageRank.stream('crypto_graph')
    yield nodeId, score
    
    with gds.util.asNode(nodeId) as c, score
    match (c)-[:COMPLAIN_TO]->(co:Company)
    
    return 
        co.name as Company,
        
        coalesce(c.consumer_complaint_narrative, "No Complaint Found") as Complaint_Narrative,
        
        round(score, 2) as Score
    order by score desc
    limit 1
    """)
    
    data = [record.data() for record in result]
    
    #cleanup
    tx.run("call gds.graph.drop('crypto_graph', false)")
    
    return data


with driver.session() as session:
    data = session.execute_read(get_the_main_complaint)
    
    if not data:
        print("didn't find anything.")
    else:
        row = data[0]
        
        print("\n" + "+"*80)
        print(f" COMPANY: {row['Company']}")
        print("+" * 80)
        print(" THE COMPLAINT NARRATIVE:")
        print("+" * 80)
        
        #textwrap makes long text readable in the terminal
        wrapper = textwrap.TextWrapper(width=80)
        print(wrapper.fill(text=row['Complaint_Narrative']))
        
        print("+"*80)


++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
 COMPANY: Fig Tech Inc.
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
 THE COMPLAINT NARRATIVE:
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
I am submitting this complaint to document my attempt to resolve an alleged debt
with Fig Loans. I am proposing a settlement of 50 % of the alleged balance of
{$120.00}, amounting to {$61.00}, with the condition that upon receipt of this
payment, Fig Loans agrees to delete the associated account from all credit
reporting agencies. I am seeking confirmation from Fig Loans regarding their
willingness to accept this settlement offer and to remove the negative tradeline
from my credit reports upon payment.
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++


## The most influential complaint among the auto loan companies:

In [20]:
def get_the_main_complaint(tx):
    #ensure we don't overwrite any graph
    tx.run("call gds.graph.drop('auto_loan_graph', false)")

    
    #loading the graph for auto loan companies
    tx.run("""
    call gds.graph.project.cypher(
      'auto_loan_graph',
      'match (c:Complaint)-[:COMPLAIN_TO]->(co:Company)
       where toLower(co.name) contains "drivetime"
          or toLower(co.name) contains "hyundai capital"
          or toLower(co.name) contains "southern auto finance"
       return id(c) as id',
      'match (c1)-[s:SIMILAR]-(c2)
       where s.similarity_score > 0.6
       return id(c1) as source, id(c2) as target',
      { validateRelationships: false }
    )
    """)
    
    #running pagerank
    result = tx.run("""
    call gds.pageRank.stream('auto_loan_graph')
    yield nodeId, score
    
    with gds.util.asNode(nodeId) as c, score
    match (c)-[:COMPLAIN_TO]->(co:Company)
    
    return 
        co.name as Company,
        
        coalesce(c.consumer_complaint_narrative, "No Complaint Found") as Complaint_Narrative,
        
        round(score, 2) as Score
    order by score desc
    limit 1
    """)
    
    data = [record.data() for record in result]
    
    #cleanup
    tx.run("call gds.graph.drop('auto_loan_graph', false)")
    
    return data


with driver.session() as session:
    data = session.execute_read(get_the_main_complaint)
    
    if not data:
        print("didn't find anything.")
    else:
        row = data[0]
        
        print("\n" + "+"*80)
        print(f" COMPANY: {row['Company']}")
        print("+" * 80)
        print(" THE COMPLAINT NARRATIVE:")
        print("+" * 80)
        
        #textwrap makes long text readable in the terminal
        wrapper = textwrap.TextWrapper(width=80)
        print(wrapper.fill(text=row['Complaint_Narrative']))
        
        print("+"*80)


++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
 COMPANY: DriveTime
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
 THE COMPLAINT NARRATIVE:
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
In //, I purchased a used from DriveTime and was financed through . The total
cost of the loan was over {$50000.00} with an interest rate of on a vehicle
currently worth only {$9000.00} {$12000.00}. I made a {$200.00} down payment and
was signed into a loan with biweekly payments of {$310.00}. The finance charge
alone is over {$25000.00}, and the total repayment amount is {$48000.00}. This
loan structure is excessive, predatory, and deeply unfair. I was not given clear
or reasonable alternatives. In //, I contacted asking for hardship support, a
refinance, or settlement option due to financial hardship. I was told they do
not offer refinancing or true hardship programs only temporary payment
extension

In [21]:
# Timestamp (End)
current_datetime = datetime.now()
formatted_time = current_datetime.strftime("%Y-%m-%d_%H:%M:%S")
print(formatted_time)

end_time = time.time()
elapsed_seconds = end_time - start_time

# Convert elapsed seconds to minutes and seconds
minutes = int(elapsed_seconds // 60)
seconds = elapsed_seconds % 60

print(f"Program elapsed time: {minutes} minutes and {seconds:.2f} seconds")

2026-01-08_22:09:04
Program elapsed time: 0 minutes and 0.64 seconds
